In [2]:
# extract_pvscan_dates_to_csv.py

import os
import csv
from datetime import datetime
import xml.etree.ElementTree as ET

# Root folders you mentioned
ROOT_FOLDERS = [
    "/Volumes/DOM_MASTER/159DCE",
    "/Volumes/DOM_MASTER/160DCE",
    "/Volumes/DOM_MASTER/162DCE",
    "/Volumes/DOM_MASTER/164DCE",
]

# Output CSV path (adjust if you want it elsewhere)
OUTPUT_CSV = "/Users/domchom/Desktop/pvscan_dates.csv"

def find_xml_files(root_folders):
    for root in root_folders:
        if not os.path.exists(root):
            print(f"Warning: folder does not exist, skipping: {root}")
            continue

        for dirpath, dirnames, filenames in os.walk(root):
            for fname in filenames:
                # Skip macOS resource fork files like ._foo.xml
                if fname.startswith("._"):
                    continue

                if fname.lower().endswith(".xml"):
                    yield os.path.join(dirpath, fname)



def extract_pvscan_datetime(xml_path):
    """
    Parse the XML and extract the date from the <PVScan> tag.
    Example:
      <PVScan version="5.4.64.700" date="9/27/2023 11:51:27 AM" notes="">
    Returns a datetime object, or None if not found / parse error.
    """
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # The root element itself is usually PVScan
        # But to be safe, we search for it.
        # If the root is PVScan, this will just return [root].
        pvscan = root if root.tag == "PVScan" else root.find("PVScan")
        if pvscan is None:
            # Sometimes namespaces appear; handle that by checking tag suffix
            for child in root.iter():
                if child.tag.endswith("PVScan"):
                    pvscan = child
                    break

        if pvscan is None:
            print(f"Warning: No PVScan tag in {xml_path}")
            return None

        date_str = pvscan.attrib.get("date")
        if not date_str:
            print(f"Warning: No 'date' attribute in PVScan tag for {xml_path}")
            return None

        # From your example: "9/27/2023 11:51:27 AM"
        # This format string should match that exactly.
        dt = datetime.strptime(date_str, "%m/%d/%Y %I:%M:%S %p")
        return dt

    except Exception as e:
        print(f"Error parsing {xml_path}: {e}")
        return None


def main():
    rows = []

    for xml_path in find_xml_files(ROOT_FOLDERS):
        dt = extract_pvscan_datetime(xml_path)
        if dt is None:
            continue

        # Derive which of the 4 roots this file belongs to
        folder_root = next((root for root in ROOT_FOLDERS if xml_path.startswith(root)), "")

        rows.append({
            "folder_root": folder_root,
            "xml_path": xml_path,
            "relative_path": os.path.relpath(xml_path, folder_root) if folder_root else xml_path,
            "pvscan_date_raw": dt.strftime("%m/%d/%Y %I:%M:%S %p"),
            "datetime_iso": dt.isoformat(),
            "date": dt.strftime("%Y-%m-%d"),
            "time": dt.strftime("%H:%M:%S"),
        })

    # Sort by datetime if desired
    rows.sort(key=lambda r: r["datetime_iso"])

    # Write CSV
    fieldnames = [
        "folder_root",
        "xml_path",
        "relative_path",
        "pvscan_date_raw",
        "datetime_iso",
        "date",
        "time",
    ]

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"Done. Wrote {len(rows)} rows to {OUTPUT_CSV}")


if __name__ == "__main__":
    main()


Done. Wrote 102 rows to /Users/domchom/Desktop/pvscan_dates.csv
